In [6]:
# Homework 06

In [34]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [35]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import statistics
from itertools import repeat
from functools import reduce
from operator import mul
from math import log

In [36]:
# Fetch the dataset
breast_cancer_data = fetch_ucirepo(id=14)

# Data (as pandas DataFrames)
X = breast_cancer_data.data.features
y = breast_cancer_data.data.targets

breast_cancer_df = pd.concat([X, y], axis=1)

In [37]:
USE_PREPRUNING = True  # Whether to apply pre-pruning
MAX_DEPTH = 5         # Maximum depth of the tree
MIN_TRAIN_SIZE = 10   # Minimum number of training samples to split
MIN_INFO_GAIN = 0     # Minimum information gain to split
USE_POSTPRUNING = True  # Whether to apply post-pruning

In [38]:
# List of all feature names (all columns except the last one, 'Class')
feature_names = list(breast_cancer_df.columns)[:-1]

In [39]:
def compute_class_entropy(data: pd.DataFrame) -> float:
    """
    Compute the entropy of the target (Class) distribution in the given DataFrame.
    
    Parameters:
    data (pd.DataFrame): The dataset.
    
    Returns:
    float: The entropy of the class distribution.
    """
    class_counts = data["Class"].value_counts(normalize=True)
    return -sum(prob * log(prob, 2) for prob in class_counts if prob > 0)

In [40]:
def compute_attribute_entropy(data: pd.DataFrame, attribute: str) -> float:
    """
    Compute the entropy of the dataset when splitting by a specific attribute.
    
    Parameters:
    data (pd.DataFrame): The dataset.
    attribute (str): Name of the feature to consider for splitting.
    
    Returns:
    float: The total entropy after splitting by the given attribute.
    """
    total_entropy = 0.0
    attribute_values = data[attribute].unique()
    attribute_counts = data[attribute].value_counts(normalize=True)

    for attr_value in attribute_values:
        subset = data[data[attribute] == attr_value]
        value_entropy = compute_class_entropy(subset)
        total_entropy += value_entropy * attribute_counts.get(attr_value, 0)
    return total_entropy

In [41]:
def compute_information_gain(data: pd.DataFrame, attribute: str) -> float:
    """
    Compute the Information Gain (IG) of splitting the dataset on a given attribute.
    
    IG = H(Class) - H(Class | Attribute)
    
    Parameters:
    data (pd.DataFrame): The dataset.
    attribute (str): Name of the feature to consider for splitting.
    
    Returns:
    float: The information gain.
    """
    return compute_class_entropy(data) - compute_attribute_entropy(data, attribute)

In [42]:
def find_best_split_attribute(data: pd.DataFrame, attributes: list) -> str:
    """
    Determine the best attribute to split on based on the highest information gain.
    
    Parameters:
    data (pd.DataFrame): The dataset.
    attributes (list): A list of attribute names.
    
    Returns:
    str: The name of the attribute providing the highest information gain.
    """
    info_gains = {attr: compute_information_gain(data, attr) for attr in attributes}
    return max(info_gains, key=info_gains.get)

In [43]:
def build_decision_tree(data: pd.DataFrame, current_depth: int = 0) -> dict:
    """
    Recursively build a decision tree using entropy and information gain.
    
    Parameters:
    data (pd.DataFrame): The dataset used to build the tree.
    current_depth (int): The current depth of the tree. Defaults to 0.
    
    Returns:
    dict: A tree node represented as a dictionary.
    """
    # If we have only one row or we reached pre-pruning limits, create a leaf.
    if (
        data.shape[0] == 1 or
        (USE_PREPRUNING and (current_depth > MAX_DEPTH or data.shape[0] < MIN_TRAIN_SIZE))
    ):
        return {
            "node_type": "leaf",
            "class": data["Class"].mode().iloc[0]
        }

    # Find the best attribute to split
    attribute_to_split = find_best_split_attribute(data, feature_names)

    # If the information gain is below the threshold, create a leaf.
    if USE_PREPRUNING and compute_information_gain(data, attribute_to_split) < MIN_INFO_GAIN:
        return {
            "node_type": "leaf",
            "class": data["Class"].mode().iloc[0]
        }

    # Otherwise, build a decision node.
    majority_class = data["Class"].mode()[0]
    unique_values = data[attribute_to_split].unique()

    return {
        "node_type": "decision",
        "attribute": attribute_to_split,
        "class_majority": majority_class,
        "decisions": {
            str(val): build_decision_tree(
                data[data[attribute_to_split] == val],
                current_depth + 1
            )
            for val in unique_values
        }
    }

In [44]:
def predict_class(decision_tree: dict, row: pd.Series) -> str:
    """
    Predict the class label for a single example using the decision tree.
    
    Parameters:
    decision_tree (dict): The trained decision tree.
    row (pd.Series): A single example.
    
    Returns:
    str: The predicted class label.
    """
    if decision_tree["node_type"] == "leaf":
        return decision_tree["class"]

    if decision_tree["node_type"] == "decision":
        attribute_value = str(row[decision_tree["attribute"]])
        next_node = decision_tree["decisions"].get(attribute_value)
        if next_node is None:
            # If the value wasn't seen during training
            return decision_tree["class_majority"]
        else:
            return predict_class(next_node, row)

    return "ERROR"

In [45]:
def is_correct_prediction(row: pd.Series, decision_tree: dict) -> bool:
    """
    Check if the decision tree's prediction matches the actual class of a row.
    
    Parameters:
    row (pd.Series): The data row containing features and the true 'Class'.
    decision_tree (dict): The decision tree used for prediction.
    
    Returns:
    bool: True if prediction matches the actual class, False otherwise.
    """
    predicted = predict_class(decision_tree, row)
    actual = row["Class"]
    return predicted == actual

In [46]:
def compute_accuracy(decision_tree: dict, test_data: pd.DataFrame) -> float:
    """
    Compute the accuracy of the decision tree on a given test dataset.
    
    Parameters:
    decision_tree (dict): The decision tree used for prediction.
    test_data (pd.DataFrame): The dataset to test.
    
    Returns:
    float: The ratio of correct predictions.
    """
    if test_data.shape[0] == 0:
        return 0.0
    correct_predictions = sum(is_correct_prediction(row, decision_tree) for _, row in test_data.iterrows())
    return correct_predictions / test_data.shape[0]

In [47]:
def apply_reduced_error_pruning(decision_tree: dict, validation_data: pd.DataFrame) -> dict:
    """
    Apply reduced error pruning on the decision tree.
    
    Parameters:
    decision_tree (dict): The decision tree to prune.
    validation_data (pd.DataFrame): The dataset used for validating pruning.
    
    Returns:
    dict: The pruned decision tree.
    """
    if decision_tree["node_type"] == "leaf":
        return decision_tree  # Can't prune further.

    if not decision_tree["node_type"] == "decision":
        return decision_tree

    # Recursively prune children
    should_prune_this_node = True
    for value_key, subtree in decision_tree["decisions"].items():
        decision_tree["decisions"][value_key] = apply_reduced_error_pruning(subtree, validation_data)
        if decision_tree["decisions"][value_key]["node_type"] != "leaf":
            should_prune_this_node = False

    # Check if converting this node to a leaf does not reduce accuracy
    if should_prune_this_node:
        # Convert to leaf
        new_leaf = {
            "node_type": "leaf",
            "class": decision_tree.get("class", decision_tree.get("class_majority", None))
        }

        current_accuracy = compute_accuracy(decision_tree, validation_data)
        leaf_accuracy = compute_accuracy(new_leaf, validation_data)

        # If accuracy is the same or better, prune!
        if leaf_accuracy >= current_accuracy:
            return new_leaf

    return decision_tree

In [48]:
def train_decision_tree(train_data: pd.DataFrame, validation_data: pd.DataFrame) -> dict:
    """
    Build a decision tree and optionally apply post-pruning.
    
    Parameters:
    train_data (pd.DataFrame): The training dataset.
    validation_data (pd.DataFrame): The validation dataset (for pruning).
    
    Returns:
    dict: A trained decision tree.
    """
    tree = build_decision_tree(train_data)
    if USE_POSTPRUNING:
        tree = apply_reduced_error_pruning(tree, validation_data)
    return tree

In [49]:
def separate_by_class(data: pd.DataFrame) -> list:
    """
    Group the dataset by unique class labels.
    
    Parameters:
    data (pd.DataFrame): The dataset.
    
    Returns:
    list: A list of dictionaries, each containing 'class_title' and 'class_table'.
    """
    class_names = data["Class"].unique()
    output = []
    for cls in class_names:
        output.append({
            "class_title": cls,
            "class_table": data[data["Class"] == cls]
        })
    return output

In [50]:
def split_train_test(data: pd.DataFrame, train_ratio: float = 0.8) -> (pd.DataFrame, pd.DataFrame):
    """
    Perform a stratified split of the dataset into training and testing sets.
    
    Parameters:
    data (pd.DataFrame): The dataset.
    train_ratio (float): Proportion of data to include in the training set.
    
    Returns:
    (pd.DataFrame, pd.DataFrame): (training_set, testing_set)
    """
    groups = separate_by_class(data)
    train_records = []
    test_records = []

    for group in groups:
        class_table = group["class_table"]
        indices = list(range(class_table.shape[0]))
        random.shuffle(indices)
        split_index = int(len(indices) * train_ratio)

        train_indices = indices[:split_index]
        test_indices = indices[split_index:]

        train_records.extend(class_table.iloc[train_indices].to_dict('records'))
        test_records.extend(class_table.iloc[test_indices].to_dict('records'))

    return pd.DataFrame(train_records), pd.DataFrame(test_records)

In [51]:
def create_stratified_folds(data: pd.DataFrame, k: int) -> list:
    """
    Create stratified folds for K-Fold cross-validation.
    
    Parameters:
    data (pd.DataFrame): The dataset.
    k (int): Number of folds.
    
    Returns:
    list: A list of pd.DataFrame, each representing a fold.
    """
    groups = separate_by_class(data)
    folds = [[] for _ in range(k)]

    for group in groups:
        class_table = group["class_table"].reset_index(drop=True)
        entries_per_fold = max(1, class_table.shape[0] // k)
        indices = list(range(class_table.shape[0]))
        random.shuffle(indices)

        current_fold = 0
        current_count = 0

        for idx in indices:
            folds[current_fold].append(class_table.iloc[idx])
            current_count += 1
            if current_count == entries_per_fold and current_fold < k - 1:
                current_fold += 1
                current_count = 0

    return [pd.DataFrame(fold) for fold in folds]

In [52]:
def perform_k_fold_cv(data: pd.DataFrame, k: int) -> None:
    """
    Perform K-Fold cross-validation on the given data.
    
    Parameters:
    data (pd.DataFrame): The dataset.
    k (int): Number of folds.
    
    Returns:
    None. Prints the accuracy for each fold and summary statistics.
    """
    print(f"------ Performing {k}-Fold Cross-Validation ------")
    folds = create_stratified_folds(data, k)
    accuracies = []

    for index in range(k):
        # Combine all folds except the current one to form the training set
        train_folds = folds[:index] + folds[index+1:]
        train_data = pd.concat(train_folds)
        train_data = train_data.set_axis(feature_names + ["Class"], axis=1)

        # The current fold is the testing set
        test_data = folds[index]
        test_data = test_data.set_axis(feature_names + ["Class"], axis=1)

        # Train a tree and compute accuracy on the current fold
        decision_tree = train_decision_tree(train_data, test_data)
        accuracy = compute_accuracy(decision_tree, test_data)
        accuracies.append(accuracy)
        print(f"[FOLD {index}] Accuracy: {accuracy:.2%}")

    print(f"Average acuracy: {statistics.mean(accuracies):.2%}")
    print(f"Standard deviation: {statistics.stdev(accuracies):.2%}")
    print(f"------ Validation completed ------")

In [55]:
def handle_missing_values(data: pd.DataFrame, use_third_option: bool = False) -> pd.DataFrame:
    """
    Handle missing values in the dataset.

    If use_third_option is True, replaces missing values with a special placeholder 'k'.
    Otherwise, fills missing values with the most frequent value in each column.

    Parameters:
    data (pd.DataFrame): The original dataset.
    use_third_option (bool): Whether to use a placeholder for missing values.

    Returns:
    pd.DataFrame: The dataset with missing values handled.
    """
    result = data.copy(deep=True)

    if use_third_option:
        # Use missing value as a distinct category
        result = result.fillna('k')
    else:
        # Replace missing values with the most frequent value (mode) of each column.
        for column in result.columns:
            mode_value = result[column].mode().iloc[0]
            # Assign the result of fillna directly back to the column
            result[column] = result[column].fillna(mode_value)

    return result


In [62]:
# Example usage of the entire flow.
random.seed()

# Handle missing values.
complete_df = handle_missing_values(breast_cancer_df, use_third_option=False)

# Update global parameters.
USE_POSTPRUNING = True
USE_PREPRUNING = True
MAX_DEPTH = 10
MIN_TRAIN_SIZE = 5
MIN_INFO_GAIN = 0.1

# Split data into train/test
train_data, test_data = split_train_test(complete_df)

# Train a tree on the training data and evaluate on training data itself
tree_for_train = train_decision_tree(train_data, train_data)
train_accuracy = compute_accuracy(tree_for_train, train_data)
print(f"Train set accuracy: {train_accuracy:.2%}")

# Perform 10-Fold cross-validation on the training set
perform_k_fold_cv(train_data, 10)

# Evaluate on the test set
tree_for_test = train_decision_tree(train_data, test_data)
test_accuracy = compute_accuracy(tree_for_test, test_data)
print(f"Test set accuracy: {test_accuracy:.2%}")

Train set accuracy: 70.18%
------ Performing 10-Fold Cross-Validation ------
[FOLD 0] Accuracy: 72.73%
[FOLD 1] Accuracy: 72.73%
[FOLD 2] Accuracy: 72.73%
[FOLD 3] Accuracy: 72.73%
[FOLD 4] Accuracy: 72.73%
[FOLD 5] Accuracy: 54.55%
[FOLD 6] Accuracy: 72.73%
[FOLD 7] Accuracy: 72.73%
[FOLD 8] Accuracy: 72.73%
[FOLD 9] Accuracy: 56.67%
Average acuracy: 69.30%
Standard deviation: 7.24%
------ Validation completed ------
Test set accuracy: 70.69%
